In [ ]:
from pathlib import Path
import torchaudio
from torchaudio.transforms import MelSpectrogram, AmplitudeToDB
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

class LibriSpeech(Dataset):
  def __init__(self, base_path, set_type):
    self.mel_transform = MelSpectrogram(sample_rate=16000, n_fft=400, n_mels=80)
    self.db_transform = AmplitudeToDB("power", 80.0)

    self.root_dir = Path(f"{base_path}/{set_type}")
    files = [p for p in self.root_dir.rglob("*") if p.is_file() and not p.name.endswith(".txt")]
    self.audio_paths = list(sorted(files))
    self.transcript_cache = {}

  def get_transcript(self, audio_path):
    name, _ = audio_path.split(".")
    s, c, i = name.split("-")[:3]
    transcript_path = f"{self.root_dir}/{s}/{c}/{s}-{c}.trans.txt"

    if transcript_path in self.transcript_cache:
      lines = self.transcript_cache[transcript_path]
      return lines[int(i)][len(name):]

    with open(transcript_path, "r") as file:
      lines = file.read().split("\n")
      self.transcript_cache[transcript_path] = lines
      return lines[int(i)][len(name):]

  def __len__(self):
    return len(self.audio_paths)

  def __getitem__(self, idx):
    waveform, sample_rate = torchaudio.load(self.audio_paths[idx])
    assert sample_rate == 16000 # As desribed by the dataset
    spectrogram = self.db_transform(self.mel_transform(waveform))
    transcript = self.get_transcript(self.audio_paths[idx])
    return spectrogram, transcript

def plot_spectrogram(spectrogram):
   _, ax = plt.subplots(1, 1)
   ax.set_title("Mel Spectrogram of Audio Sample")
   ax.set_ylabel("Mel frequency")
   ax.set_xlabel("Mel bands")
   ax.imshow(spectrogram, origin="lower", aspect="auto", interpolation="nearest")

dataset = LibriSpeech(".cache/LibriSpeech", "dev-clean")
loader = DataLoader(dataset, shuffle=True, batch_size=1)
plot_spectrogram(list(next(loader))[0][0])

: 